# Component 1 | live frozen U-Net inference

Runs the exact V3 weights, not a new training run. Included example is a noncommercial CubiCasa diagnostic floorplan. Outputs are **fresh pixel predictions pending review**. This notebook does not run YOLO, extract confidence-filtered components, calibrate a Twin or optimize an event layout. Saved example cell outputs are local execution evidence, not a Google Colab run.


## Retained curated demo: cubi_03304

A 12-candidate search did not find a materially better full-frame result. The higher-scoring cubi_00412 was rejected as a replacement after full-size visual QA because of severe exterior artifacts. The existing simple cubi_03304 is retained, **not presented as corrected or flawless**. Its selected-example IoU is not unbiased model performance. Raw errors, every trial and cubi_03153 remain preserved; historical training/evaluation records are unchanged.


In [ ]:
from pathlib import Path
import os, sys, json
MYDRIVE_PACKAGE = "COMPONENT_1_PP1_COLAB_DELIVERY_V1"  # change if your Drive folder name differs
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount('/content/drive')  # normal user authorization, no credentials in this notebook
    PACKAGE_ROOT = Path('/content/drive/MyDrive') / MYDRIVE_PACKAGE
else:
    PACKAGE_ROOT = Path(os.environ.get('PP1_COLAB_PACKAGE', Path.cwd())).resolve()
    if PACKAGE_ROOT.name == '01_NOTEBOOKS':
        PACKAGE_ROOT = PACKAGE_ROOT.parent
assert (PACKAGE_ROOT / 'SOURCE_PROVENANCE.json').is_file(), 'Choose the extracted package root, not the ZIP or notebook folder.'
sys.path.insert(0, str(PACKAGE_ROOT / '09_RUNTIME'))
from portable_unet import read, verify_assets
def show_png(path):
    from PIL import Image
    import matplotlib.pyplot as plt
    with Image.open(path) as image:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.imshow(image)
    ax.axis('off'); fig.tight_layout(); plt.show()
print('Execution location:', 'Colab (remote run by you)' if IN_COLAB else 'Local Python; NOT a remote Colab run')
print(verify_assets(PACKAGE_ROOT))


## Dependencies and runtime

Colab normally supplies Torch/Torchvision. Keep its matching pair. Only missing/mismatched auxiliary dependencies are installed, with the existing Torch versions constrained. If installation or imports fail, stop and report the actual error; do not load substitute weights. Package versions are pinned to the tested auxiliary stack, but remote availability has not been verified. GPU is optional; CPU fallback is supported and tested locally.


In [ ]:
import importlib.metadata as metadata
import subprocess
INSTALL_IF_NEEDED = IN_COLAB  # never installs packages during the local verification
required = {'segmentation-models-pytorch': '0.5.0', 'timm': '1.0.30'}
missing = []
for package, version in required.items():
    try:
        found = metadata.version(package)
    except metadata.PackageNotFoundError:
        found = None
    if found != version:
        missing.append(f'{package}=={version}')
for package, requirement in [('numpy', 'numpy>=1.26,<3'), ('Pillow', 'Pillow>=10,<13'), ('matplotlib', 'matplotlib>=3.8,<4')]:
    try: metadata.version(package)
    except metadata.PackageNotFoundError: missing.append(requirement)
if missing:
    if not INSTALL_IF_NEEDED:
        raise RuntimeError('Missing/incompatible packages: ' + str(missing))
    # Keep the Colab-provided compatible Torch/Torchvision pair; do not upgrade it incidentally.
    import tempfile
    constraints = Path(tempfile.mkdtemp()) / 'constraints.txt'
    constraints.write_text('torch==' + metadata.version('torch') + '\n' + 'torchvision==' + metadata.version('torchvision') + '\n')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-c', str(constraints), *missing])
import torch, torchvision, segmentation_models_pytorch, numpy, PIL, matplotlib
print({p: metadata.version(p) for p in ['torch', 'torchvision', 'segmentation-models-pytorch', 'timm', 'numpy', 'Pillow', 'matplotlib']})
print('A successful local dependency check is not verified remote Colab compatibility.')


In [ ]:
import time
from portable_unet import load_model, predict, CHECKPOINT_SHA256
torch.set_num_threads(4)
DEVICE = os.environ.get('PP1_INFERENCE_DEVICE', 'auto')  # auto selects GPU when available, else CPU
model, device, checkpoint_metadata = load_model(PACKAGE_ROOT, device=DEVICE)
print('Checkpoint SHA256:', CHECKPOINT_SHA256)
print('Device:', device)
print('Checkpoint metadata:', checkpoint_metadata)


## One actual forward pass

Original RGB → aspect-preserving bilinear 512 letterbox → RGB/255 → ImageNet normalization → [1,3,512,512] → four logits → argmax. Crop only padding, then nearest-neighbour resize labels back to source pixels. Timer includes one forward, argmax and CPU transfer, with GPU synchronization when needed; excludes model loading and preprocessing. No warm-up or throughput claim.


In [ ]:
import tempfile
source = PACKAGE_ROOT / '05_INFERENCE_INPUTS/UNET_CURATED_DEMO/cubi_03304/SOURCE.png'
run_dir = Path(tempfile.mkdtemp(prefix='pp1-inference-')) / 'example'
prediction = predict(model, device, source, run_dir)
print(json.dumps(prediction, indent=2))


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from portable_unet import CLASSES, COLORS
from matplotlib.patches import Patch
fig, axes = plt.subplots(1, 3, figsize=(14, 6))
for ax, filename, title in zip(axes, ['SOURCE_DECODED.png', 'PREDICTION_COLOR.png', 'OVERLAY.png'], ['Original publisher raster', 'Fresh U-Net labels', 'Fresh prediction overlay']):
    with Image.open(run_dir / filename) as image: ax.imshow(image)
    ax.set_title(title); ax.axis('off')
fig.legend(handles=[Patch(color=[v/255 for v in color], label=name) for name, color in zip(CLASSES, COLORS)], loc='lower center', ncol=4)
fig.suptitle('Curated cubi_03304 | UNEDITED prediction; exterior errors retained | review required')
plt.tight_layout(rect=(0, .07, 1, .95)); plt.show()


## Optional user upload

Default is off. Upload only a drawing you are permitted to process with Google Colab. This notebook supports a single PNG/JPEG, not CAD/PDF. A new drawing has no supplied ground truth; no accuracy is generated. KNOX/Ravenswood notebook masks would be fresh U-Net outputs, not historical API results. Their API path additionally runs opening YOLO, confidence thresholding and component extraction, so candidate counts must not be promised identical.


In [ ]:
RUN_USER_UPLOAD = False
if RUN_USER_UPLOAD:
    if not IN_COLAB:
        raise RuntimeError('Upload widget is available only in Google Colab; use predict() with a local permitted input otherwise.')
    from google.colab import files
    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError('Select exactly one PNG or JPEG')
    name, data = next(iter(uploads.items()))
    if Path(name).suffix.lower() not in ['.png', '.jpg', '.jpeg'] or len(data) > 20 * 1024 * 1024:
        raise ValueError('Only PNG/JPEG up to 20 MiB is supported')
    upload_root = Path(tempfile.mkdtemp(prefix='pp1-user-'))
    user_source = upload_root / Path(name).name
    user_source.write_bytes(data)
    uploaded_result = predict(model, device, user_source, upload_root / 'prediction')
    show_png(upload_root / 'prediction/OVERLAY.png')
    print(json.dumps(uploaded_result, indent=2))
else:
    print('Optional user upload skipped; included example above actually ran.')


## Limits

No metric scale, ground-truth accuracy for a new source, accepted venue boundary, physical inventory, planner approval, access safety, event layout or novelty is inferred. Use the separate reviewed Twin workflow for any later spatial decisions. Download fresh files from the displayed temporary run directory if required; source evidence and the checkpoint are never overwritten.
